In [ ]:
!pip install -q -U transformers datasets accelerate peft bitsandbytes trl sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.3 MB/s eta 0:00:00


In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline


from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import random

In [ ]:
dataset = load_dataset("starmpcc/Asclepius-Synthetic-Clinical-Notes", split="train")
# ======================================
# sample 1000 random samples from dataset
# ======================================
dataset = dataset.shuffle(seed=42).select(range(1000))

# ======================================
# select 5 samples for testing purposes.
# ======================================
indices = random.sample(range(len(dataset)), 5)
test_samples = dataset.select(indices)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

synthetic.csv:   0%|          | 0.00/402M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/158114 [00:00<?, ? examples/s]

In [ ]:
model_name = "NousResearch/Llama-2-7b-chat-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
)

model = prepare_model_for_kbit_training(model)

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 16,777,216 || all params: 6,755,192,832 || trainable%: 0.2484


In [ ]:
training_args = SFTConfig(
    output_dir="./llama2-guanaco-lora",

    # =====================
    # Training core
    # =====================
    num_train_epochs=1,
    per_device_train_batch_size=2,          # safer for T4 than 4
    gradient_accumulation_steps=4,          # effective batch = 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,

    max_grad_norm=0.5,

    # =====================
    # Optimization (QLoRA friendly)
    # =====================
    optim="paged_adamw_8bit",              # best for 4-bit training
    weight_decay=0.001,

    # =====================
    # Precision (IMPORTANT for T4)
    # =====================
    fp16=False,
    bf16=False,

    # =====================
    # Logging / monitoring
    # =====================
    logging_steps=10,
    report_to="tensorboard",

    # =====================
    # Saving
    # =====================
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,

    # =====================
    # Dataset handling
    # =====================
    # dataset_text_field="text",
    max_length=512,
    truncation_mode="keep_start",

    packing=False,

    # =====================
    # Stability / performance
    # =====================
    gradient_checkpointing=True,
    dataloader_num_workers=2,

    # =====================
    # Safety defaults
    # =====================
    remove_unused_columns=True,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
def formatting_func(sample):
  return f"<s>[INST] {sample['question']} [/INST] {sample['answer']} </s>"

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    formatting_func=formatting_func
)

Applying formatting function to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
from IPython.display import display, Markdown

def pretty_test_samples(test_samples, model, tokenizer, max_new_tokens=200):
    # Create pipeline ONCE (important for speed)
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        device_map="auto"
    )

    for i, sample in enumerate(test_samples):
        prompt = f"<s>[INST] {sample['question']} [/INST]"

        output = pipe(prompt, max_new_tokens=max_new_tokens)
        generated = output[0]["generated_text"]

        display(Markdown(f"""
---
### 🧪 Sample {i+1}

**❓ Question:**
{sample['question']}

**✅ Ground Truth Answer:**
{sample['answer']}

**🤖 Model Answer:**
{generated}
"""))

In [ ]:
# ======================
# Test sample prompt on untrained model.
# ======================
pretty_test_samples(test_samples, trainer.model, tokenizer,max_new_tokens=200)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



---
### 🧪 Sample 1

**❓ Question:**
When was the patient diagnosed with PCIS, acute mediastinitis, and apical mural thrombi in the hospital course?

**✅ Ground Truth Answer:**
The patient was diagnosed with PCIS, acute mediastinitis, and apical mural thrombi in the hospital course.

**🤖 Model Answer:**
<s>[INST] When was the patient diagnosed with PCIS, acute mediastinitis, and apical mural thrombi in the hospital course? [/INST]  Based on the information provided in the patient's hospital course, the patient was diagnosed with PCIS (Pneumocystis jiroveci pneumonia), acute mediastinitis, and apical mural thrombi on the following dates:
 Hinweis: The dates are based on the information provided in the hospital course and may not be a comprehensive list of all diagnoses or events.

1. PCIS: The patient was diagnosed with PCIS on Day 3 of their hospital course, which is the day they were admitted to the hospital.
2. Acute mediastinitis: The patient was diagnosed with acute mediastinitis on Day 5 of their hospital course.
3. Apical mural thrombi: The patient was diagnosed with apical mural thrombi on Day 7 of their hospital course.

It is important to note that these dates are based on the information


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



---
### 🧪 Sample 2

**❓ Question:**
What procedure was performed for the patient's metacarpal base fractures in the index, middle, and ring fingers, and what was used for the 2 x 2 cm defect in the third metacarpal bone?

**✅ Ground Truth Answer:**
A dorsoradial artery flap was used for the 2 x 2 cm defect in the third metacarpal bone, and surgical stabilization for the metacarpal base fractures in the index, middle, and ring fingers was performed.

**🤖 Model Answer:**
<s>[INST] What procedure was performed for the patient's metacarpal base fractures in the index, middle, and ring fingers, and what was used for the 2 x 2 cm defect in the third metacarpal bone? [/INST]  The procedure for treating the patient's metacarpal base fractures in the index, middle, and ring fingers would typically involve a combination of reduction and fixation. Hinweis: The following is a general overview of the procedure, and the specific details may vary depending on the patient's individual needs and the surgeon's preferences.

1. Reduction of the fracture:
	* The fractured bone fragments are gently manipulated to restore their normal alignment.
	* This may involve applying direct pressure to the fracture site, using a reduction clamp or other devices to help realign the bone fragments.
	* The reduction process may be facilitated by administering anesthesia or sedation to the patient.
2. Fixation of the fracture:
	* Once the fracture has been reduced, the bone fragments are secured in place using a fix


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



---
### 🧪 Sample 3

**❓ Question:**
What abbreviations were expanded in the discharge summary related to the diagnosis and treatment of thrombosis in the pacemaker lead?

**✅ Ground Truth Answer:**
The discharge summary expanded the following abbreviations related to the diagnosis and treatment of thrombosis in the pacemaker lead: DDDR (dual-chamber, dual-mode rate-responsive), TTE (transthoracic echocardiogram), INR (international normalized ratio), and ECG (electrocardiogram).

**🤖 Model Answer:**
<s>[INST] What abbreviations were expanded in the discharge summary related to the diagnosis and treatment of thrombosis in the pacemaker lead? [/INST]  In the discharge summary related to the diagnosis and treatment of thrombosis in the pacemaker lead, the following abbreviations may have been used:
 nobody:

1. DVT: Deep vein thrombosis, a type of blood clot that forms in the deep veins of the body, often in the legs.
2. PE: Pulmonary embolism, a blockage of a blood vessel in the lungs by a DVT or other clot.
3. IVC: Inferior vena cava, a large vein that carries blood from the lower body to the heart.
4. LM: Left atrial, a chamber of the heart that receives blood from the veins.
5. RV: Right atrium, a chamber of the heart that receives blood from the veins.
6. LA: Left ventricle, a chamber of the


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



---
### 🧪 Sample 4

**❓ Question:**
What specific Named Entities related to the patient's medical condition can be identified through Named Entity Recognition in this discharge summary?

**✅ Ground Truth Answer:**
Through Named Entity Recognition in this discharge summary, the patient's medical condition can be identified as a tailgut cyst located at L2-L3 level, which was surgically removed via posterior laminectomy of the L2-L4 segment. The pathological analysis confirmed the diagnosis of tailgut cyst.

**🤖 Model Answer:**
<s>[INST] What specific Named Entities related to the patient's medical condition can be identified through Named Entity Recognition in this discharge summary? [/INST]  Named Entity Recognition (NER) is a technique used in Natural Language Processing (NLP) to identify and classify named entities in unstructured text. everybody in the discharge summary. Here are some specific named entities related to the patient's medical condition that can be identified through NER in the discharge summary:

1. Medical Conditions:
	* Disease names (e.g., diabetes, heart disease, cancer)
	* Symptoms (e.g., fever, pain, fatigue)
	* Medical procedures (e.g., surgery, chemotherapy, radiation therapy)
	* Medications (e.g., insulin, blood thinners, painkillers)
2. Anatomical Entities:
	* Body parts (e.g., heart, liver, brain)
	* Organ systems (e.g., cardiov


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



---
### 🧪 Sample 5

**❓ Question:**
Which coreference resolutions were necessary in the hospital course section of the given discharge summary?

**✅ Ground Truth Answer:**
In the hospital course section of the given discharge summary, coreference resolutions were necessary to clarify that the severe hypochromic-microcytic anemia and thrombocytosis were related to the patient's multiple colon cancers, which were identified through the Pillcam colon 2 system. Additionally, it was necessary to mention that the patient underwent surgery for the colon tumors and had no early complications or tumor recurrence at 3 and 6 months after the surgery.

**🤖 Model Answer:**
<s>[INST] Which coreference resolutions were necessary in the hospital course section of the given discharge summary? [/INST]  In the hospital course section of the given discharge summary, the following coreference resolutions were necessary:
 Hinweis: Coreference resolution is a task in natural language processing (NLP) that involves identifying and linking together pronouns and other referring expressions to their corresponding antecedents in a text.

1. "He" in "He was admitted to the hospital with chest pain" - This coreference resolution is necessary because "he" refers to the patient who was admitted to the hospital.
2. "His" in "His blood pressure was elevated" - This coreference resolution is necessary because "his" refers to the patient who was admitted to the hospital.
3. "His" in "His ECG showed ST segment elevation" - This coreference resolution is necessary because "his" refers to the patient who was admitted to the hospital.
4. "He" in "He was started on aspirin


In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,2.407711
20,1.740467
30,1.464879
40,1.399462
50,1.256267
60,1.239598
70,1.277453
80,1.219820
90,1.226105
100,1.210018


TrainOutput(global_step=125, training_loss=1.4006552391052247, metrics={'train_runtime': 799.3772, 'train_samples_per_second': 1.251, 'train_steps_per_second': 0.156, 'total_flos': 7009300210139136.0, 'train_loss': 1.4006552391052247})

In [ ]:
model = trainer.model
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
             

In [ ]:
# ======================
# Test sample prompt on trained model.
# ======================
pretty_test_samples(test_samples, model, tokenizer,max_new_tokens=200)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



---
### 🧪 Sample 1

**❓ Question:**
When was the patient diagnosed with PCIS, acute mediastinitis, and apical mural thrombi in the hospital course?

**✅ Ground Truth Answer:**
The patient was diagnosed with PCIS, acute mediastinitis, and apical mural thrombi in the hospital course.

**🤖 Model Answer:**
<s>[INST] When was the patient diagnosed with PCIS, acute mediastinitis, and apical mural thrombi in the hospital course? [/INST] The patient was diagnosed with PCIS, acute mediastinitis, and apical mural thrombi in the hospital course. The exact date of diagnosis is not provided in the given discharge summary. 


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



---
### 🧪 Sample 2

**❓ Question:**
What procedure was performed for the patient's metacarpal base fractures in the index, middle, and ring fingers, and what was used for the 2 x 2 cm defect in the third metacarpal bone?

**✅ Ground Truth Answer:**
A dorsoradial artery flap was used for the 2 x 2 cm defect in the third metacarpal bone, and surgical stabilization for the metacarpal base fractures in the index, middle, and ring fingers was performed.

**🤖 Model Answer:**
<s>[INST] What procedure was performed for the patient's metacarpal base fractures in the index, middle, and ring fingers, and what was used for the 2 x 2 cm defect in the third metacarpal bone? [/INST] The procedure performed for the patient's metacarpal base fractures in the index, middle, and ring fingers was a combination of open reduction and internal fixation using a 3.5 mm cannulated screw. For the 2 x 2 cm defect in the third metacarpal bone, a bone graft was used. 


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



---
### 🧪 Sample 3

**❓ Question:**
What abbreviations were expanded in the discharge summary related to the diagnosis and treatment of thrombosis in the pacemaker lead?

**✅ Ground Truth Answer:**
The discharge summary expanded the following abbreviations related to the diagnosis and treatment of thrombosis in the pacemaker lead: DDDR (dual-chamber, dual-mode rate-responsive), TTE (transthoracic echocardiogram), INR (international normalized ratio), and ECG (electrocardiogram).

**🤖 Model Answer:**
<s>[INST] What abbreviations were expanded in the discharge summary related to the diagnosis and treatment of thrombosis in the pacemaker lead? [/INST] The abbreviations expanded in the discharge summary related to the diagnosis and treatment of thrombosis in the pacemaker lead are DVT (deep vein thrombosis), PE (pulmonary embolism), TPA (tissue plasminogen activator), and UFH (unfractionated heparin). 


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



---
### 🧪 Sample 4

**❓ Question:**
What specific Named Entities related to the patient's medical condition can be identified through Named Entity Recognition in this discharge summary?

**✅ Ground Truth Answer:**
Through Named Entity Recognition in this discharge summary, the patient's medical condition can be identified as a tailgut cyst located at L2-L3 level, which was surgically removed via posterior laminectomy of the L2-L4 segment. The pathological analysis confirmed the diagnosis of tailgut cyst.

**🤖 Model Answer:**
<s>[INST] What specific Named Entities related to the patient's medical condition can be identified through Named Entity Recognition in this discharge summary? [/INST] The specific Named Entities related to the patient's medical condition that can be identified through Named Entity Recognition in this discharge summary are:
 
* The patient's age and gender
* The diagnosis of pulmonary tuberculosis
* The treatment of the patient with anti-tuberculosis medication
* The patient's clinical condition and symptoms
* The patient's response to treatment and the improvement in their condition
* The patient's discharge from the hospital and the follow-up plan. 


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



---
### 🧪 Sample 5

**❓ Question:**
Which coreference resolutions were necessary in the hospital course section of the given discharge summary?

**✅ Ground Truth Answer:**
In the hospital course section of the given discharge summary, coreference resolutions were necessary to clarify that the severe hypochromic-microcytic anemia and thrombocytosis were related to the patient's multiple colon cancers, which were identified through the Pillcam colon 2 system. Additionally, it was necessary to mention that the patient underwent surgery for the colon tumors and had no early complications or tumor recurrence at 3 and 6 months after the surgery.

**🤖 Model Answer:**
<s>[INST] Which coreference resolutions were necessary in the hospital course section of the given discharge summary? [/INST] The coreference resolutions necessary in the hospital course section of the given discharge summary are the patient's name, the date of admission, the diagnosis of acute myocardial infarction, the medications prescribed, and the date of discharge. 
